# Session 8: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over an alternative investments knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

Enter your OpenAI API Key: ········


In [3]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using the Alternative Investments Handbook - a comprehensive resource covering alternative investment strategies, reinsurance, private equity, real estate, commodities, and portfolio management.

### Data Preparation

We'll load the investments handbook as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/AlternativeInvestmentsHandbook.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
investment_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [5]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(investment_docs)}")
print(f"\nExample chunk:\n{investment_docs[0]}")

Raw documents: 1
Split chunks: 77

Example chunk:
page_content='The Alternative Investments Handbook
A Practical Guide to Non-Traditional Asset Classes and Risk Premiums

PART 1: FOUNDATIONS OF ALTERNATIVE INVESTING

Chapter 1: What Are Alternative Investments?' metadata={'source': 'data/AlternativeInvestmentsHandbook.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "investments_handbook".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    investment_docs,
    embeddings,
    location=":memory:",
    collection_name="investments_handbook",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [9]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

"Key due diligence steps for alternative investments include:\n\n1. **Assessing Investment Strategy and Process:** Confirm that the strategy is clearly defined, repeatable, and has a sound theoretical basis for generating returns.\n\n2. **Evaluating Manager Track Record:** Review the manager's performance across multiple market cycles to ensure consistency with the stated strategy.\n\n3. **Analyzing Risk Management:** Examine the risk controls in place, such as loss limits, leverage management, and worst-case scenario planning.\n\n4. **Understanding the Source of Return:** Determine whether returns come from risk premiums, alpha, structural advantages, or other sources.\n\n5. **Liquidity and Redemption Terms:** Review lock-up periods, redemption notice requirements, gates, and how they align with the investment strategy and your liquidity needs.\n\n6. **Fee Structure:** Analyze management and performance fees, ensuring they are appropriate and properly aligned with performance.\n\n7. *

In [12]:
naive_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits primarily because its returns are fundamentally uncorrelated with traditional financial markets such as equities and bonds. This low correlation stems from the fact that reinsurance risk is driven by natural catastrophes like hurricanes, earthquakes, and severe weather events—hazards that are not linked to economic conditions. As a result, incorporating reinsurance into a portfolio can reduce overall volatility and enhance risk-adjusted returns. Additionally, reinsurance offers diversifiable risk across different geographies and peril types, further strengthening its role as a diversification tool.'

In [13]:
naive_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include:\n\n1. **Leveraged Buyouts (LBOs):** Acquiring mature companies with a combination of equity and debt, aiming to improve operations, reduce costs, and grow revenue before exiting through sale or IPO.\n\n2. **Growth Equity:** Investing in established, profitable companies that need capital to expand, such as for geographic growth, new products, or acquisitions.\n\n3. **Distressed Investing:** Buying debt or equity of financially troubled companies at significant discounts, with value creation through restructuring, operational improvements, or asset sales.\n\n4. **Secondaries:** Purchasing existing private equity fund interests from other investors to diversify investments, often at a discount to net asset value.\n\n5. **Event-Driven Strategies:** Investing around corporate events like mergers, acquisitions, restructurings, or bankruptcies, including merger arbitrage.\n\nThese strategies are designed to target different types of 

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [14]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(investment_docs)

We'll construct the same chain - only changing the retriever.

In [15]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [16]:
bm25_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

'The key due diligence steps for alternative investments include examining the following areas:\n\n1. Investment Strategy and Process: Ensure the strategy is clearly defined, repeatable, and based on a sound theoretical basis for generating returns. Also, evaluate how the strategy has performed historically.\n\n2. Track Record: Review the manager’s performance across multiple market cycles to assess consistency with the stated strategy.\n\n3. Risk Management: Investigate what controls are in place to limit losses, how leverage is managed, and consider worst-case scenarios.\n\n4. Source of Return: Identify whether returns come from risk premiums, alpha, or structural advantages.\n\n5. Liquidity and Redemption Terms: Understand how liquid the investment is and the conditions for redemption.\n\n6. Fee Structure: Review fees and how they align with performance.\n\n7. Market and Environment Fit: Consider how the investment fits within your overall portfolio.\n\n8. Tax Implications: Evaluate

In [17]:
bm25_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits because it has a low correlation with traditional financial markets. For example, investments in catastrophe bonds and insurance-linked securities (ILS) tend to be unaffected by general market movements or economic conditions. Such assets often perform positively or remain stable even during widespread market downturns, as their returns are tied to specific events like natural disasters rather than broader market trends. This helps reduce overall portfolio volatility and enhances risk-adjusted returns by adding assets that behave differently from traditional equities and bonds.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing typically include buyouts, venture capital, and growth equity. These strategies involve investing directly in private companies or in takeovers of public companies to delist them from public markets. Private equity investments aim to generate returns through active management, improving company performance, and eventually realizing gains upon exit through sales or initial public offerings (IPOs).'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

**Example query:** *"What are the key features of a catastrophe bond (CAT bond)?"*

BM25 is better here because "CAT bond" is a rare, domain-specific abbreviation. Embedding models may dilute its meaning by averaging it toward semantically adjacent terms like "bond" or "insurance," pulling in irrelevant documents. BM25 rewards exact term matches, so it reliably surfaces documents that contain "CAT bond" verbatim. Generally, BM25 wins for queries with precise technical terms, acronyms, or proper nouns where exact matching matters more than semantic similarity.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [20]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever

from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [21]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

'The key due diligence steps for alternative investments include:\n\n1. Examining the investment strategy to understand the source of returns, such as risk premium, alpha, or structural advantages.\n2. Assessing the liquidity and redemption terms to determine how easily you can exit the investment.\n3. Reviewing the fee structure and evaluating how fees are aligned with performance.\n4. Evaluating the manager’s track record across different market environments to gauge their experience and reliability.\n5. Understanding the risk management systems in place to mitigate potential risks.\n6. Considering how the investment fits within the overall portfolio context.\n7. Analyzing the tax implications associated with the investment.\n\nThese steps are vital because alternative investments are often less transparent and more complex than traditional investments.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits by offering returns that are largely uncorrelated with traditional financial markets like equities and bonds. Since reinsurance investments are driven primarily by natural catastrophes and weather events, their performance does not depend on economic conditions. This low correlation helps reduce overall portfolio risk and smooth out returns. Additionally, reinsurance risk can be spread across different geographies and peril types, further enhancing diversification.'

In [24]:
contextual_compression_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

"The main strategies for private equity investing include:\n\n1. **Leveraged Buyouts (LBOs):** Acquiring mature companies using a combination of equity and debt financing. The goal is to improve the company's operations, reduce costs, and increase revenue before exiting through a sale or IPO.\n\n2. **Direct Investment or Take-Private Investments:** Investing directly in private companies or acquiring public companies to take them private, with the aim of enhancing their growth and governance for a profitable exit.\n\n3. **Event-Driven Strategies:** Investing around corporate events such as mergers, acquisitions, restructurings, and bankruptcies. For example, merger arbitrage involves buying the target company and shorting the acquirer during announced deals.\n\nIf you need more details on any of these strategies, feel free to ask!"

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!


In [27]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [29]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [30]:
multi_query_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

"The key due diligence steps for alternative investments include:\n\n1. **Examining the Investment Strategy and Process:** Ensure the strategy is clearly defined, repeatable, and theoretically capable of delivering the expected returns.\n\n2. **Assessing the Manager's Track Record:** Review the manager’s performance across multiple market cycles to verify consistency with the stated strategy.\n\n3. **Evaluating Risk Management Systems:** Identify controls in place to limit losses, manage leverage, and prepare for worst-case scenarios.\n\n4. **Analyzing the Source of Return:** Understand whether returns are generated from risk premiums, alpha, or structural advantages.\n\n5. **Liquidity and Redemption Terms:** Review liquidity profiles, including lock-up periods, redemption notice requirements, and gate provisions, to ensure they align with the investment strategy and investor needs.\n\n6. **Fee Structure and Alignment:** Analyze fee arrangements, including management and performance fe

In [31]:
multi_query_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits primarily because its returns are driven by natural catastrophe risks such as weather and seismic events, which are largely uncorrelated with traditional financial markets like equities and bonds. This low correlation helps reduce overall portfolio volatility. Additionally, reinsurance risk can be spread across different geographies and peril types, further enhancing diversification. As a result, including reinsurance investments can help balance portfolio risk and potentially improve risk-adjusted returns.'

In [32]:
multi_query_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include:\n\n1. Leveraged Buyouts (LBOs): Acquiring mature companies using a combination of equity and debt, with the aim of improving operations, reducing costs, and growing revenue for a profitable exit.\n\n2. Growth Equity: Investing in established, profitable companies that need capital to expand, often for geographic growth, new product development, or acquisitions.\n\n3. Distressed Investing: Buying debt or equity of financially troubled companies at significant discounts, with value creation through restructuring, operational improvements, or asset sales.\n\n4. Secondaries: Purchasing existing private equity fund interests from other investors to achieve diversification and potential discounts.\n\n5. Venture Capital (VC): Investing in early-stage, high-growth companies, particularly in sectors like technology and healthcare.\n\nThese strategies often vary in terms of risk, liquidity, and involvement, but all aim to generate attrac

### Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

A single query encodes just one semantic perspective, so relevant documents phrased differently may fall outside its nearest-neighbor neighborhood. By generating multiple reformulations—paraphrases, synonyms, different levels of specificity—the retriever casts a wider net across the embedding space. Documents that miss the original query's vector but match a reformulation are still returned, increasing the likelihood that all relevant passages are captured. The unique union of results across all queries is what raises recall without requiring any changes to the underlying index.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [35]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [36]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="investments_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="investments_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [37]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [38]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [39]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [40]:
parent_document_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

'The key due diligence steps for alternative investments include:\n\n1. **Assessment of Investment Strategy and Process:** Ensure the strategy is clearly defined, repeatable, and based on sound theoretical principles that justify expected returns.\n\n2. **Evaluation of Track Record:** Review the manager’s performance over multiple market cycles to confirm consistency with the stated strategy.\n\n3. **Risk Management Review:** Understand what controls are in place to limit losses, how leverage is managed, and evaluate worst-case scenarios to ensure risks are appropriately mitigated.\n\n4. **Operational Infrastructure Inspection:** Confirm there is adequate separation between portfolio management and back-office operations, and verify the involvement of independent service providers such as custodians, auditors, and administrators.\n\n5. **Analysis of Fee Structure:** Examine whether fees are reasonable relative to the value added, and whether the fee arrangements (including high-water m

In [41]:
parent_document_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits primarily because its returns have near-zero correlation with traditional financial markets such as equities and bonds. This is due to the fact that reinsurance risk is driven by natural catastrophes like hurricanes, earthquakes, and floods, which are not influenced by economic or market conditions. As a result, incorporating reinsurance into an investment portfolio can help reduce overall risk and improve risk-adjusted returns. Additionally, reinsurance offers attractive risk premiums associated with the severity and frequency of natural disasters, and its risks can be spread across different geographies and peril types, further enhancing diversification.'

In [42]:
parent_document_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include:\n\n1. **Leveraged Buyouts (LBOs):** Acquiring mature companies using a combination of equity and debt financing. The goal is to improve operations, reduce costs, and grow revenue before exiting through a sale or IPO. Historically, LBOs have generated returns of 2-3 times the invested capital.\n\n2. **Growth Equity:** Investing in established, profitable companies that need capital to expand through geographic growth, new product development, or acquisitions.\n\n3. **Distressed Investing:** Purchasing the debt or equity of financially troubled companies at significant discounts. Value creation comes from financial restructuring, operational improvements, or asset sales.\n\n4. **Secondaries:** Buying existing private equity fund interests from other investors. This strategy offers diversification across vintage years and strategies, often at a discount to net asset value.\n\nThese strategies each serve different investment object

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [44]:
from langchain_classic.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [45]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [46]:
ensemble_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

"The key due diligence steps for alternative investments include:\n\n1. **Assessing the Investment Strategy and Process:**\n   - Is the strategy clearly defined and repeatable?\n   - What is the theoretical basis for expected returns?\n\n2. **Evaluating the Manager's Track Record:**\n   - How has the manager performed across different market cycles?\n   - Are the returns consistent with the stated strategy?\n\n3. **Analyzing Risk Management Systems:**\n   - What controls are in place to limit losses?\n   - How is leverage managed?\n   - What are the worst-case scenarios?\n\n4. **Reviewing Operational Infrastructure:**\n   - Is there separation of duties between portfolio management and back-office operations?\n   - Are there independent service providers like custodians, auditors, and administrators?\n\n5. **Understanding Fee Structures:**\n   - Are fees reasonable relative to the value added?\n   - Are they aligned with investor interests through mechanisms such as high-water marks an

In [47]:
ensemble_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits primarily because its returns are largely uncorrelated with traditional financial markets such as equities and bonds. This is because reinsurance investments are driven by natural catastrophes and weather events—like hurricanes, earthquakes, and storms—that are not linked to economic conditions or market movements. \n\nBy including reinsurance and insurance-linked securities (ILS) such as catastrophe bonds (cat bonds), investors can reduce overall portfolio volatility and enhance risk-adjusted returns. These assets offer attractive risk premiums due to the catastrophic and low-frequency nature of the risks involved, and they can be spread across different geographies and peril types, further enhancing diversification. Additionally, because the underlying risks are event-driven and do not depend on market performance, reinsurance investments can act as a natural hedge against economic downturns, providing more stability to an inve

In [48]:
ensemble_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include:\n\n1. Leveraged Buyouts (LBOs): Acquiring mature companies using a combination of equity and debt financing with the goal of improving operations, reducing costs, and growing revenue before exiting through sale or IPO. Historically, LBOs have generated returns of 2-3 times the invested capital.\n\n2. Growth Equity: Investing in established, profitable companies that need capital for expansion, such as geographic growth, product development, or acquisitions.\n\n3. Distressed Investing: Purchasing debt or equity of financially troubled companies at significant discounts, with value creation through financial restructuring, operational improvements, or asset sales.\n\n4. Secondaries: Buying existing private equity fund interests from other investors, offering diversification across vintages and strategies, often at a discount to net asset value.\n\nThese strategies collectively form the core approaches used by private equity firms

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [49]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [50]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [51]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="investments_handbook_semantic_chunks"
)

We'll use naive retrieval for this example.

In [52]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [53]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [54]:
semantic_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

'The key due diligence steps for alternative investments include:\n\n1. **Evaluate the Investment Strategy and Process**\n   - Ensure the strategy is clearly defined and repeatable.\n   - Understand the theoretical basis for generating returns, such as risk premiums or structural advantages.\n\n2. **Assess the Manager’s Track Record**\n   - Review performance over multiple market cycles.\n   - Verify that returns are consistent with the stated strategy.\n\n3. **Examine Risk Management Systems**\n   - Check for controls to limit losses.\n   - Understand leverage management and worst-case scenarios.\n\n4. **Review Operational Infrastructure**\n   - Confirm adequate separation of duties between portfolio management and back-office operations.\n   - Identify independent service providers for custody, administration, and auditing.\n\n5. **Analyze Fee Structure**\n   - Ensure fees are reasonable relative to the value added.\n   - Verify alignment of interests through provisions like high-wat

In [55]:
semantic_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits because its returns are largely uncorrelated with traditional financial markets like equities and bonds. This is because reinsurance investments are driven primarily by natural catastrophe and weather events (such as hurricanes, earthquakes, floods) rather than economic or market conditions. As a result, gains or losses from reinsurance tend to occur independently of stock or bond market movements, helping to reduce overall portfolio volatility. Additionally, reinsurance offers attractive risk premiums and can provide steady income through premium payments, further enhancing diversification and risk-adjusted returns when included in a broader investment portfolio.'

In [56]:
semantic_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

"The main strategies for private equity investing include:\n\n1. Leveraged Buyouts (LBOs): Acquiring mature companies using a combination of equity and debt financing with the goal of improving operations, reducing costs, and growing revenue before exiting through a sale or IPO. Historically, this strategy has generated returns of 2-3 times the invested capital.\n\n2. Growth Equity: Investing in established companies that need capital to expand, helping them grow further while typically holding the investments for longer periods.\n\n3. Venture Capital: Investing in early-stage, high-growth-potential companies, especially in sectors like technology and healthcare, with staged funding such as seed, Series A, and later rounds.\n\n4. Distressed Investing: Purchasing debt or equity of financially troubled companies at discounts, then creating value through restructuring or asset sales.\n\n5. Secondaries: Buying existing private equity fund interests from other investors, offering diversific

### Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

With short, repetitive sentences, inter-sentence distances will be uniformly small—nearly everything is semantically similar—so the percentile threshold may rarely trigger a break, producing very large chunks that lump unrelated Q&A pairs together. Alternatively, if the threshold is too sensitive, it may over-split, creating single-sentence chunks that lack useful context.

To address this, you'd want to raise the breakpoint percentile to force more splits, or switch to a `gradient` thresholding method which detects sharp local changes in similarity rather than relying on a global distribution. You could also pre-process by grouping each question-answer pair as a logical unit before feeding into the chunker, bypassing the sentence-level splitting entirely.

---

# Breakout Room Part #2

### Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [ ]:
### YOUR CODE HERE